# NeuroScan AI: Entrenamiento Acelerado en Google Colab Pro con MLflow
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/03_training_colab_mlflow.ipynb)

Este notebook implementa la **Etapa 3**:
1. Detección y activación de aceleración por GPU (NVIDIA A100 / V100 / L4) y **Mixed Precision (FP16)**.
2. Sincronización automática con Google Drive para respaldar checkpoints y base de datos SQLite de MLflow.
3. Ejecución de **5 Experimentos de Transfer Learning**:
   - **Run 1:** ResNet50 Base Congelada (LR 1e-3, Dropout 0.3)
   - **Run 2:** ResNet50 Base Congelada (LR 1e-4, Dropout 0.5)
   - **Run 3:** ResNet50 Fine-Tuning capa 143 (LR 1e-5, Dropout 0.3)
   - **Run 4 (Variante 1):** ResNet50 Fine-Tuning profundo capa 120 + Cosine Decay
   - **Run 5 (Variante 2):** EfficientNetB2 Transfer Learning (Comparación Multiarquitectura)


In [1]:
# 0. Inicialización Automática de Entorno (Google Colab / Local)
import os
import sys

# Detección y preparación autónoma para Google Colab
if 'google.colab' in sys.modules:
    # 1. Clonar el repositorio público en /content si no existe
    if not os.path.exists('/content/m3-proyecto-final-databricks-ia'):
        print("🚀 Clonando repositorio NeuroScan AI en Google Colab...")
        !git clone https://github.com/EduardCY/m3-proyecto-final-databricks-ia.git /content/m3-proyecto-final-databricks-ia

    # 2. Posicionarse en la carpeta raíz del proyecto
    %cd /content/m3-proyecto-final-databricks-ia

    # 3. Instalar dependencias requeridas para Colab (MLflow, etc.)
    !pip install -q -r requirements-colab.txt

# 4. Asegurar que la raíz del proyecto está en el PYTHONPATH
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Entorno preparado en: {PROJECT_ROOT}")


/content/m3-proyecto-final-databricks-ia
✅ Entorno preparado en: /content/m3-proyecto-final-databricks-ia


In [2]:
# 1. Configuración de Entorno y Detección de GPU
import os
import sys
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Aceleradora Detectada: {gpus}")
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("🚀 Mixed Precision FP16 habilitada para Tensor Cores.")
else:
    print("⚠️ No se detectó GPU. Ejecutando en CPU.")


TensorFlow: 2.20.0
✅ GPU Aceleradora Detectada: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
🚀 Mixed Precision FP16 habilitada para Tensor Cores.


In [3]:
# 2. Montar Google Drive para Respaldo Resiliente
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/m3_project'
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/backups", exist_ok=True)
    print(f"✅ Google Drive conectado en: {DRIVE_DIR}")
except Exception as e:
    print(f"Ejecutando en entorno local sin montaje de Colab Drive: {e}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive conectado en: /content/drive/MyDrive/m3_project


In [4]:
# 3. Verificación y Descarga del Dataset de Resonancias
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path
from scripts.download_dataset import verify_dataset_structure, generate_smoke_test_dataset

data_raw = Path("data/raw")
data_raw.mkdir(parents=True, exist_ok=True)

if not verify_dataset_structure(data_raw):
    print("📥 Descargando dataset Brain Tumor MRI (~165 MB)...")
    zip_url = "https://github.com/masoudnickparvar/brain-tumor-mri-dataset/archive/refs/heads/main.zip"
    zip_tmp = Path("data/dataset_archive.zip")
    try:
        urllib.request.urlretrieve(zip_url, zip_tmp)
        with zipfile.ZipFile(zip_tmp, 'r') as zip_ref:
            zip_ref.extractall("data/temp_extracted")

        extracted_root = next(Path("data/temp_extracted").glob("brain-tumor-mri-dataset-*"))
        for item in extracted_root.iterdir():
            dest = data_raw / item.name
            if dest.exists():
                if dest.is_dir():
                    shutil.rmtree(dest)
                else:
                    dest.unlink()
            shutil.move(str(item), str(data_raw))
        shutil.rmtree("data/temp_extracted", ignore_errors=True)
        zip_tmp.unlink(missing_ok=True)
        print("✅ Dataset descargado y descomprimido exitosamente en data/raw/")
    except Exception as e:
        print(f"⚠️ Fallo en descarga remota: {e}. Generando dataset de contingencia para pruebas...")
        generate_smoke_test_dataset(data_raw, samples_per_class=30)
else:
    print(f"✅ Dataset disponible en: {data_raw}")


✅ Dataset disponible en: data/raw


In [5]:
# 4. Auditoría y Construcción de Datasets tf.data
import pandas as pd
from src.dataset import build_manifest, create_stratified_split
from src.config import BATCH_SIZE, CLASSES, SPLIT_MANIFEST_PATH
from src.preprocessing import create_tf_dataset
from src.train import train_and_log_run

# Generar manifiesto con rutas del entorno actual
df_manifest = build_manifest(data_dir=data_raw, use_cache=False)
df_split = create_stratified_split(df_manifest, output_path=SPLIT_MANIFEST_PATH)
print(f"✅ Manifiesto y Split generados con {len(df_split)} imágenes:")
print(df_split.groupby(["split", "label_name"]).size().unstack())

train_ds = create_tf_dataset(df_split, split_name="train", batch_size=BATCH_SIZE, model_type="resnet50")
val_ds = create_tf_dataset(df_split, split_name="val", batch_size=BATCH_SIZE, model_type="resnet50")
test_ds = create_tf_dataset(df_split, split_name="test", batch_size=BATCH_SIZE, model_type="resnet50")
print("✅ Pipelines tf.data listos para GPU.")


ImportError: cannot import name 'create_stratified_split' from 'src.dataset' (/content/m3-proyecto-final-databricks-ia/src/dataset.py)

### 4. Ejecución de los 5 Experimentos en MLflow


In [ ]:
# Run 1: ResNet50 Frozen Base (LR 1e-3, Dropout 0.3)
m1, h1, met1 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_1_ResNet50_Frozen_1e3",
    backbone_name="resnet50",
    epochs=12, learning_rate=1e-3, dropout=0.3,
    base_trainable=False
)


In [ ]:
# Run 2: ResNet50 Frozen Base (LR 1e-4, Dropout 0.5)
m2, h2, met2 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_2_ResNet50_Frozen_1e4",
    backbone_name="resnet50",
    epochs=12, learning_rate=1e-4, dropout=0.5,
    base_trainable=False
)


In [ ]:
# Run 3: ResNet50 Fine-Tuning capa 143 (LR 1e-5, Dropout 0.3)
m3, h3, met3 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_3_ResNet50_FineTune_143",
    backbone_name="resnet50",
    epochs=15, learning_rate=1e-5, dropout=0.3,
    base_trainable=True, fine_tune_at=143
)


In [ ]:
# Run 4 (Variante 1): ResNet50 Fine-Tuning profundo capa 120 + Cosine Decay
m4, h4, met4 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_4_ResNet50_DeepFineTune_120_Cosine",
    backbone_name="resnet50",
    epochs=15, learning_rate=3e-5, dropout=0.3,
    base_trainable=True, fine_tune_at=120,
    use_cosine_decay=True
)


In [ ]:
# Run 5 (Variante 2): EfficientNetB2 Transfer Learning (Comparación Multiarquitectura)
train_eff = create_tf_dataset(df_split, split_name="train", batch_size=BATCH_SIZE, model_type="efficientnetb2")
val_eff = create_tf_dataset(df_split, split_name="val", batch_size=BATCH_SIZE, model_type="efficientnetb2")
test_eff = create_tf_dataset(df_split, split_name="test", batch_size=BATCH_SIZE, model_type="efficientnetb2")

m5, h5, met5 = train_and_log_run(
    train_dataset=train_eff, val_dataset=val_eff, test_dataset=test_eff,
    run_name="Run_5_EfficientNetB2_Transfer",
    backbone_name="efficientnetb2",
    epochs=12, learning_rate=1e-3, dropout=0.3,
    base_trainable=False
)


### 5. Sincronización Final de Resultados a Google Drive


In [ ]:
from scripts.sync_drive import backup_to_gdrive
backup_to_gdrive()
print("✅ Todos los checkpoints, modelos y la base de datos de MLflow fueron respaldados en Google Drive.")
